In [52]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [53]:
data = pd.read_csv("clean_data.csv")

In [54]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17201 entries, 0 to 17200
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Net_Metrekare       17201 non-null  int64  
 1   Brüt_Metrekare      17201 non-null  float64
 2   Oda_Sayısı          17201 non-null  float64
 3   Bulunduğu_Kat       17201 non-null  object 
 4   Binanın_Yaşı        17201 non-null  object 
 5   Isıtma_Tipi         17201 non-null  object 
 6   Fiyat               17201 non-null  float64
 7   Şehir               17201 non-null  object 
 8   Binanın_Kat_Sayısı  17201 non-null  int64  
 9   Kullanım_Durumu     17201 non-null  object 
 10  Banyo_Sayısı        17201 non-null  float64
dtypes: float64(4), int64(2), object(5)
memory usage: 1.4+ MB


## Model oluşturmaya hazırlık

Veri setimiziden hedef sütunu ayırma

In [55]:
x = data.drop("Fiyat",axis=1)
y = data["Fiyat"]


Kategorik verileri encode etme

In [56]:
kategorik_veri = x.select_dtypes(include=["object"]).columns
x = pd.get_dummies(x,columns=kategorik_veri,drop_first=True)

x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17201 entries, 0 to 17200
Columns: 115 entries, Net_Metrekare to Kullanım_Durumu_Mülk Sahibi Oturuyor
dtypes: bool(110), float64(3), int64(2)
memory usage: 2.5 MB


Eğitim ve Test verisi olarak ayırma

In [57]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.33,random_state=42)

Ölçeklendirme 

Scaling önemli olan modeller:

    Linear Regression

    SVR

    KNN

    Logistic Regression

Scaling gerekmeyen modeller:

    Decision Tree

    Random Forest

    XGBoost

In [58]:
sc = StandardScaler()

x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)

## Model oluşturma

Kütüphaneler

In [59]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


## Model Seçme

### Metrik anlamları:
##### 1. MAE (Ortalama Mutlak Hata)

###### Detay: Tahminlerin ile gerçek değerler arasındaki farkların basit ortalamasıdır. Hataların negatif veya pozitif olması fark etmez.

###### Örnek: Bir araba fiyatı tahmin modelinde MAE değerin 10.000 ise, modelin arabaların fiyatını ortalama 10.000 TL eksik veya fazla tahmin ediyor demektir.

##### 2. RMSE (Kök Ortalama Kare Hata)

###### Detay: Hataların önce karesi alınıp sonra ortalamasının karekökü bulunduğu için, modelin yaptığı büyük hatalara karşı çok daha hassastır ve bunları yüksek skorla cezalandırır.

###### Örnek: 100 arabanın fiyatını 1.000 TL hatayla, 1 arabanın fiyatını ise 500.000 TL hatayla tahmin edersen; MAE bu durumu pek umursamaz ama RMSE o tek büyük hata yüzünden anında fırlar. Modelin tutarsızlıklarını görmek için harikadır.

##### 3. R² (R-Kare / Modelin Açıklama Gücü)

###### Detay: Genellikle 0 ile 1 arasında değer alır. Elindeki girdilerin (özelliklerin), tahmin etmeye çalıştığın hedefteki değişimi yüzde kaç oranında açıklayabildiğini gösterir. 1'e ne kadar yakınsa model o kadar başarılıdır.

###### Örnek: R² skorun 0.85 ise, araba fiyatlarındaki değişikliğin %85'ini modelindeki verilerle (kilometre, yıl, hasar durumu vb.) doğru bir şekilde yakalayabiliyorsun demektir.

In [60]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import pandas as pd
import numpy as np

# 1. XGBoost için parametre optimizasyonu
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 300]
}


xgb_grid = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=3, scoring='r2', n_jobs=-1)
xgb_grid.fit(x_train, y_train)

print(f"En iyi parametreler: {xgb_grid.best_params_}")
print("----------------------")

# 2. Modelleri tanımlama
models = {
    "RandomForest": RandomForestRegressor(random_state=42),
    "SVR" : SVR(),
    "GradientBoostingRegressor" : GradientBoostingRegressor(random_state=42),
    "XGBRegressor_Optimize" : xgb_grid.best_estimator_
}

results = []

# 3. Model Eğitimi ve Testi
for name, model in models.items():
    # SVR scaled veri kullanır
    if name == "SVR":
        model.fit(x_train_scaled, y_train)
        y_pred = model.predict(x_test_scaled)
    else:
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append([
        name,
        round(r2,2),
        round(mae/1000,0),
        round(rmse/1000000,2)
    ])

    print(f"{name} R2: {r2:.2f}")
    print(f"{name} MAE: {mae/1000:.0f}k")
    print(f"{name} RMSE: {rmse/1000000:.2f}M")
    print("----------------------")

# Decision Tree
dt = DecisionTreeRegressor(max_depth=3, random_state=42)
dt.fit(x_train, y_train)
dt_pred = dt.predict(x_test)

r2 = r2_score(y_test, dt_pred)
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

results.append([
    "DecisionTree",
    round(r2,2),
    round(mae/1000,0),
    round(rmse/1000000,2)
])

# DataFrame
results_df = pd.DataFrame(results, columns=["Model","R2","MAE","RMSE"])

# Sırala

results_df = results_df.sort_values(by="R2", ascending=False)

results_df

En iyi parametreler: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
----------------------
RandomForest R2: 0.57
RandomForest MAE: 549k
RandomForest RMSE: 0.88M
----------------------
SVR R2: -0.05
SVR MAE: 930k
SVR RMSE: 1.37M
----------------------
GradientBoostingRegressor R2: 0.55
GradientBoostingRegressor MAE: 584k
GradientBoostingRegressor RMSE: 0.90M
----------------------
XGBRegressor_Optimize R2: 0.59
XGBRegressor_Optimize MAE: 538k
XGBRegressor_Optimize RMSE: 0.85M
----------------------


,Model,R2,MAE,RMSE
3,XGBRegressor_Optimize,0.59,538.0,0.85
0,RandomForest,0.57,549.0,0.88
2,GradientBoostingRegressor,0.55,584.0,0.90
4,DecisionTree,0.37,733.0,1.06
1,SVR,-0.05,930.0,1.37


# EN İYİ MODEL XGBRegressor


In [61]:
best_model = XGBRegressor()
best_model.fit(x_train, y_train)

y_pred_best = best_model.predict(x_test)